# Module 1 — Kinematics (runnable)

Inverse/forward kinematics, the Jacobian, and singularities for the 2-RPR machine.

## Setup
Run this first. It defines the machine and the formulas using only Python's standard library, so the notebook runs anywhere with no `pip install`.

In [ ]:
# PKM curriculum — runnable Python reference (standard library only, no installs needed)
from math import pi, sqrt, hypot

# machine defaults (identical to the simulation engine)
B, L_CLOSED, STROKE = 0.6, 0.4, 0.6
BORE, ROD = 0.040, 0.022
SUPPLY, RELIEF = 16e6, 21e6
PUMP_MAX_FLOW, RATED_FLOW, RATED_DP = 6e-4, 2.5e-4, 3.5e6
PAYLOAD = 12.0

def inverse_kinematics(x, y, b=B): return hypot(x+b, y), hypot(x-b, y)
def forward_kinematics(L1, L2, b=B):
    x = (L1**2 - L2**2)/(4*b); return x, sqrt(max(0.0, L1**2 - (x+b)**2))
def det_jacobian(x, y, b=B):
    L1, L2 = inverse_kinematics(x, y, b); return 2*b*y/(L1*L2)
def manipulability(x, y, b=B): return abs(det_jacobian(x, y, b))
def cap_area(D=BORE): return pi*D**2/4
def rod_area(D=BORE, d=ROD): return pi*(D**2 - d**2)/4
def asymmetry(D=BORE, d=ROD): return cap_area(D)/rod_area(D, d)
def valve_flow(u, dP, Qr=RATED_FLOW, dPr=RATED_DP): return u*Qr*sqrt(max(0.0, dP/dPr))
print("reference loaded — stdlib only")

### Lesson 2.1 — Inverse kinematics
Pose → leg lengths. Place the platform at (0.10, 0.70) m.

In [ ]:
L1, L2 = inverse_kinematics(0.10, 0.70)
print(f'L1={L1:.3f} m, L2={L2:.3f} m')   # expect 0.990, 0.860

### Lesson 2.1 — Worked move
Slide from (0,0.7) to (0.1,0.7); see one leg extend, one retract.

In [ ]:
before = inverse_kinematics(0.0, 0.70)
after  = inverse_kinematics(0.10, 0.70)
for i in (0,1):
    print(f'leg {i+1}: {before[i]:.3f} -> {after[i]:.3f} m  (d={1000*(after[i]-before[i]):+.0f} mm)')

### Lesson 2.2 — Forward kinematics
Leg lengths → pose. Round-trip the result above.

In [ ]:
x, y = forward_kinematics(*after)
print(f'x={x:.3f} m, y={y:.3f} m')   # expect 0.100, 0.700

### Lesson 2.3 — Reachability
A pose is reachable only if both legs lie within [L_CLOSED, L_CLOSED+STROKE].

In [ ]:
def reachable(x, y):
    return all(L_CLOSED <= L <= L_CLOSED+STROKE for L in inverse_kinematics(x, y))
for p in [(0.0,0.70),(0.10,0.70),(0.0,1.2)]:
    print(p, '->', 'reachable' if reachable(*p) else 'OUT OF RANGE')

### Lesson 3.1 — Jacobian & manipulability
The Jacobian maps platform velocity to leg-rate; manipulability measures dexterity.

In [ ]:
print('det(J) at (0.10,0.70):', round(det_jacobian(0.10,0.70),4))
print('manip  at (0.10,0.70):', round(manipulability(0.10,0.70),4))

### Lesson 3.2 — Singularities
Approach the base line (y → 0) and watch manipulability collapse toward 0.

In [ ]:
for y in [0.7, 0.4, 0.2, 0.05]:
    print(f'y={y:>4}:  manip = {manipulability(0.10, y):.4f}')